# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)


## 1. Ranked actions + reason codes

**What this playbook is:** a *review-first* queue for content/SEO reviewers in this lane — which pages a human opens first, with the reason printed next to each name. What it is NOT: an auto-action list. Every action word here is decision-support; no sentence claims that doing X will produce Y.

The queue has two rails, both licensed by committed receipts:

- **Model rail — review-first top 40.** The Week-5 logistic model's grouped out-of-fold scores (the same 21-feature construction, client-disjoint folds — its honest numbers: AUC 0.659 / P@20 0.55 vs the hand rule's 0.544 / 0.35, base rate 15.9%). The score decides the *order of reading*, nothing else.
- **Size-of-prize rail — top 10 biggest pages not already in the top 40.** The model ranks decline risk, not prize size — and this lane measured volume OPPOSITE-signed as risk (near-floor pages decline 20.8%, n=35,073; giants 7.4%, n=95), so a risk-ordered queue alone would bury the biggest prizes. A human looks at those ten regardless of score.

**Reason codes** (each printed beside the page name, each licensed by a receipt):

- `MOMENTUM_SPIKE` — this-week pace jumped to >2.0× last-week pace. The w04 calibration table: hot-spike pages declined 29.8% vs the 15.9% base (n=5,763), and the model's top permutation lever (+0.276 standardized coef). Observed association, one 11-day label.
- `CTR_LEAK` — click rate sits below the median for its position band. w04's audited cells: low-CTR pages declined +11.0 / +7.7 / +7.3 pp more than high-CTR pages in the top-3 / 4-10 / 11-30 bands (each band n≥4,367). Directional, slice-conditional.
- `SIZE_OF_PRIZE` — impressions ≥ the frame's 90th percentile. Carried on every flagged page onto either rail.
- `META_UNKNOWN` — the page's update metadata is unresolvable at decision time (80.6% of the frame). A human with real access resolves it in minutes; the staleness verdict is MIXED — this code flags a *lookup*, never an automated action.
- `RULE_RAIL` — placed on the size-of-prize rail outside the model's top 40.

Everything the model *doesn't* see: 13,037 of 13,277 decliners (98%) rank beyond 500 — the playbook therefore ships with a named blind-spot rule in section 3, because the honest use of a P@20 0.55 queue includes knowing what it misses. Observed on one slice (month = 2026-03, decision 03-20, 11-day label window): directional, decision-support.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Rebuild the frame exactly as committed (w05/w06 construction -> identical honest numbers),
# score pages with the model, assign reason codes, print the two rails + archetype tables.
%pip -q install duckdb huggingface_hub scikit-learn matplotlib
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import duckdb
import numpy as np
import pandas as pd
import json

pd.set_option("display.width", 200)
print("duckdb", duckdb.__version__)
import sklearn
print("sklearn", sklearn.__version__)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found. One-time setup: (1) accept the gate at "
        "huggingface.co/datasets/FlyRank/internship-warehouse (affiliation 'FlyRank ML Internship 2026'); "
        "(2) create a READ token; (3) make it available (setx locally or a Colab Secret). Then re-run."
    )
print("token found: present in environment (value not printed)")

from huggingface_hub import snapshot_download

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    for parent in [Path("/content"), Path.cwd()]:
        target = parent / "flyrank-ml"
        if not target.parent.exists():
            continue
        if not (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            print(f"repo not found locally; cloning the public repo into {target} ...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tanay1122/flyrank-ml.git", str(target)], check=False, capture_output=True)
        if (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return target
    raise FileNotFoundError("No flyrank-ml repo found -- run from inside the repo (or let the Colab fallback clone it).")

REPO = find_repo_root()
CACHE = REPO / "work" / "cache" / "warehouse"
LOCAL = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=["dim_clients.parquet", "dim_content.parquet", "fact_content_daily_performance/month=2026-03/*.parquet"],
    token=HF_TOKEN,
    local_dir=str(CACHE),
)
CACHE_DIR = Path(LOCAL)
DIM_CONTENT = (CACHE_DIR / "dim_content.parquet").as_posix()
FACT_MAR = (CACHE_DIR / "fact_content_daily_performance" / "month=2026-03" / "data_0.parquet").as_posix()
print("local files ready:", Path(FACT_MAR).name, "-", round(Path(FACT_MAR).stat().st_size / 1e6, 1), "MB")

con = duckdb.connect()

# --- identical frame policy to ML-07/08/09 (same slice, same floor, same drops) ---
frame = con.execute(f'''
    WITH early AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{FACT_MAR}')
        WHERE report_date <= DATE '2026-03-20' AND gsc_data_available IS TRUE
    ), aggregated AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_early_tot,
               SUM(gsc_clicks) AS clicks_early_tot,
               AVG(CASE WHEN report_date >= DATE '2026-03-14' THEN gsc_impressions END) AS pace_mean,
               AVG(CASE WHEN report_date <= DATE '2026-03-13' THEN gsc_impressions END) AS base_mean,
               AVG(gsc_avg_position) AS pos_mean,
               SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_active,
               MAX(CASE WHEN gsc_impressions > 0 THEN report_date END) AS last_imps_day
        FROM early
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND COUNT(*) >= 10
    )
    SELECT *,
           pace_mean / NULLIF(base_mean, 0) AS momentum,
           100.0 * clicks_early_tot / imp_early_tot AS click_rate_pct,
           DATE '2026-03-20' - last_imps_day AS stale_rows
    FROM aggregated
''').df()

target = con.execute(f'''
    SELECT client_hash_id, content_hash_id, AVG(gsc_impressions) AS target_mean
    FROM read_parquet('{FACT_MAR}')
    WHERE report_date >= DATE '2026-03-21' AND report_date <= DATE '2026-03-31'
          AND gsc_data_available IS TRUE
    GROUP BY 1, 2
''').df()

frame = frame.merge(target, on=["client_hash_id", "content_hash_id"], how="left")
no_outcome = int(frame["target_mean"].isna().sum())
frame = frame[frame["target_mean"].notna()].copy()
pace_ok = frame["pace_mean"] > 0
no_pace = int((~pace_ok).sum())
n_final = len(frame)
frame = frame[pace_ok].copy()
frame["decline_11d"] = (frame["target_mean"] < 0.70 * frame["pace_mean"]).astype(int)
print(f"Frame: {len(frame):,} pages | disclosed drops: {no_outcome:,} absent, {no_pace:,} no-pace | base rate {frame['decline_11d'].mean() * 100:.1f}%")

# --- decision-time metadata, identical handling ---
dim = con.execute(f'''
    SELECT client_hash_id, content_hash_id, search_volume, competition, word_count, backlinks,
           content_type, main_intent, content_created_date, content_updated_date
    FROM read_parquet('{DIM_CONTENT}')
''').df()
before_n = len(frame)
frame = frame.merge(dim, on=["client_hash_id", "content_hash_id"], how="left")
assert frame[["client_hash_id", "content_hash_id"]].drop_duplicates().shape[0] == before_n, "1:1 join broken"

DECISION = pd.Timestamp("2026-03-20")
st_meta = (DECISION - frame["content_updated_date"].fillna(frame["content_created_date"])).dt.days
frame["meta_unknown"] = ((st_meta < 0) | st_meta.isna()).astype(int)
frame["meta_stale_days"] = np.where(frame["meta_unknown"] == 1, 0, st_meta.clip(lower=0))

# --- lane feature set (identical to w05/w06) ---
frame["momentum_capped"] = frame["momentum"].clip(lower=0, upper=10)
frame["log_imp"] = np.log1p(frame["imp_early_tot"])
frame["log_click_rate"] = np.log1p(frame["click_rate_pct"].clip(lower=0))
frame["log_search"] = np.log1p(frame["search_volume"].fillna(0).clip(lower=0))
frame["log_wordcount"] = np.log1p(frame["word_count"].fillna(0).clip(lower=0))
frame["log_backlinks"] = np.log1p(frame["backlinks"].fillna(0).clip(lower=0))
frame["competition_filled"] = frame["competition"].fillna(0.0)
frame["stale_rows_capped"] = frame["stale_rows"].clip(lower=0, upper=20).astype(float)

pos_band = pd.cut(frame["pos_mean"], bins=[-1, 3, 10, 30, float("inf")], labels=["top-3", "4-10", "11-30", "31+"])
band_med = frame.assign(pos_band=pos_band).groupby("pos_band", observed=True)["click_rate_pct"].transform("median")
frame["ctr_fix_flag"] = (frame["click_rate_pct"] < band_med).astype(int)

NUMERIC_FEATURES = [
    "momentum_capped", "log_imp", "log_click_rate", "pos_mean", "stale_rows_capped", "days_active",
    "meta_stale_days", "meta_unknown", "ctr_fix_flag",
    "log_search", "competition_filled", "log_wordcount", "log_backlinks",
]
dummies = pd.get_dummies(
    frame["main_intent"].fillna("unknown").astype(str).str.lower() + "|" + frame["content_type"].fillna("unknown").astype(str).str.lower(),
    prefix="it",
)
FEATURE_NAMES = NUMERIC_FEATURES + list(dummies.columns)
X = pd.concat([frame[NUMERIC_FEATURES].astype(float), dummies.astype(float)], axis=1)
y = frame["decline_11d"].to_numpy()
groups = frame["client_hash_id"].to_numpy()
print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features (identical construction to w05/w06)")

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import sys
sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import precision_at_k

def auc(truth, scores):
    truth = pd.Series(truth).reset_index(drop=True)
    ranks = pd.Series(np.asarray(scores)).rank()
    n_pos = int((truth == 1).sum())
    n_neg = int((truth == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return float((ranks[truth == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def make_lr():
    return Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=2000, random_state=0))])

gkf = GroupKFold(n_splits=5)
oof_lr = np.full(len(frame), np.nan)
fold_aucs = []
disjoint_all = True
for i, (tr, te) in enumerate(gkf.split(X, y, groups=groups), 1):
    mdl = make_lr().fit(X.iloc[tr], y[tr])
    oof_lr[te] = mdl.predict_proba(X.iloc[te])[:, 1]
    fold_aucs.append(auc(y[te], oof_lr[te]))
    disjoint_all = disjoint_all and not (set(groups[tr]) & set(groups[te]))

assert np.isfinite(oof_lr).all(), "every page must receive exactly one OOF score"
model_order = np.argsort(-oof_lr, kind="stable")
rank_of = np.empty(len(frame), dtype=int)
rank_of[model_order] = np.arange(len(frame))

# --- cross-check vs committed receipts (identical construction must reproduce to the digit) ---
w04_sig = json.load(open(REPO / "work" / "outputs" / "w04_signal_receipt.json", encoding="utf-8"))
w04_base = json.load(open(REPO / "work" / "outputs" / "w04_baseline_receipt.json", encoding="utf-8"))
w05 = json.load(open(REPO / "work" / "outputs" / "w05_model_receipt.json", encoding="utf-8"))
lr_row = next(r for r in w05["comparison_table"] if r["design"].startswith("Logistic Regression"))
queue_auc_100 = round(100 * auc(y, oof_lr), 1)
p20_live = float(precision_at_k(y, oof_lr, 20))
xc_auc = abs(queue_auc_100 - lr_row["AUC"]) < 0.11
xc_p20 = abs(p20_live - lr_row["P@20"]) < 0.005
print(f"receipt cross-check vs committed w05 LR row: AUC {queue_auc_100} vs {lr_row['AUC']} (match = {xc_auc}); "
      f"P@20 {p20_live:.2f} vs {lr_row['P@20']} (match = {xc_p20}) | {w05['split']['clients']} clients, folds disjoint = {disjoint_all}")

# --- reason codes (mechanical, <= 03-20 data only, one line each) ---
imp_p90 = float(frame["imp_early_tot"].quantile(0.90))
arch = pd.DataFrame(index=frame.index)
arch["MOMENTUM_SPIKE"] = frame["momentum_capped"] > 2.0
arch["CTR_LEAK"] = frame["ctr_fix_flag"] == 1
arch["SIZE_OF_PRIZE"] = frame["imp_early_tot"] >= imp_p90
arch["META_UNKNOWN"] = frame["meta_unknown"] == 1

base_rate = 100 * y.mean()
arch_rows = []
for code in arch.columns:
    sel = arch[code].to_numpy()
    n_sel = int(sel.sum())
    dec = 100 * y[sel].mean() if n_sel else float("nan")
    arch_rows.append({"code": code, "n": n_sel, "decline_pct": round(dec, 1) if n_sel else None,
                      "share_pct": round(100 * n_sel / len(frame), 1)})
arch_table = pd.DataFrame(arch_rows)
arch_table["base_rate_pct"] = round(base_rate, 1)
print()
print("REASON CODES - each beside the base rate it must beat (linked receipt in title):")
print(arch_table.to_string(index=False))
print("receipts: MOMENTUM_SPIKE ~ w04 calibration hot-spike 29.8% n=5,763; CTR_LEAK ~ w04 gaps +11.0/+7.7/+7.3 pp;")
print(f"          SIZE_OF_PRIZE ~ w04 volume table (volume OPPOSITE as risk); META_UNKNOWN ~ {int(frame['meta_unknown'].sum()):,} pages, staleness verdict {w04_base['verdictA2_staleness_metadata']}")

# --- the two rails ---
top40_idx = [int(p) for p in model_order[:40]]
top40_set = set(top40_idx)
rail_idx = [int(p) for p in np.argsort(-frame["imp_early_tot"].to_numpy(), kind="stable") if p not in top40_set][:10]

row_recs = []
for p in top40_idx:
    codes = [c for c in arch.columns if arch.iloc[p][c]]
    r = frame.iloc[p]
    row_recs.append({
        "queue": "MODEL_TOP40", "rank": int(rank_of[p]) + 1,
        "content_hash_id": r["content_hash_id"], "client_hash_id": r["client_hash_id"],
        "score": round(float(oof_lr[p]), 4), "reason_codes": "+".join(codes) if codes else "NO_CODE",
        "imps_early": int(r["imp_early_tot"]), "pos_mean": round(float(r["pos_mean"]), 1),
        "decline_11d": int(y[p]),
    })
for j, p in enumerate(rail_idx, 1):
    codes = [c for c in arch.columns if arch.iloc[p][c]]
    r = frame.iloc[p]
    row_recs.append({
        "queue": "SIZE_RAIL", "rank": j,
        "content_hash_id": r["content_hash_id"], "client_hash_id": r["client_hash_id"],
        "score": round(float(oof_lr[p]), 4), "reason_codes": "+".join(codes + ["RULE_RAIL"]) if codes else "RULE_RAIL",
        "imps_early": int(r["imp_early_tot"]), "pos_mean": round(float(r["pos_mean"]), 1),
        "decline_11d": int(y[p]),
    })
queue_df = pd.DataFrame(row_recs)

top40 = queue_df[queue_df["queue"] == "MODEL_TOP40"]
code_counts = {}
for s in top40["reason_codes"]:
    for c in s.split("+"):
        code_counts[c] = code_counts.get(c, 0) + 1
code_tbl = pd.DataFrame(sorted(code_counts.items(), key=lambda kv: -kv[1]), columns=["reason_code", "in_top40"])
print()
print("MODEL RAIL - review-first top 40, reason codes the human sees (outcome column is printed for the honest read, NOT used to build the queue):")
print(code_tbl.to_string(index=False))
p40_obs = 100 * y[top40_idx].mean()
print(f"honest read: model top-40 contains {int(y[top40_idx].sum())} decliners = {p40_obs:.1f}% of those pages "
      f"(vs base rate {base_rate:.1f}% -> random 40 would hold ~{0.15 * 40:.1f}); top-20 P@ = {p20_live:.2f} (committed receipt 0.55)")
rail = queue_df[queue_df["queue"] == "SIZE_RAIL"]
print(f"SIZE-OF-PRIZE RAIL - {len(rail)} biggest pages outside the top 40 (sum {int(rail['imps_early'].sum()):,} impressions early-window):")
print(rail[["rank", "reason_codes", "imps_early", "decline_11d"]].to_string(index=False))
missed = int(((y == 1) & (rank_of > 500)).sum())
total_dec = int(y.sum())
print(f"blind-spot rule (printed every run): {missed:,} of {total_dec:,} decliners ({100 * missed / total_dec:.0f}%) rank beyond 500 - the queue is review-order, not coverage")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = REPO / "work" / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.barh(list(code_tbl["reason_code"])[::-1], list(code_tbl["in_top40"])[::-1], color="#5878a4")
ax.set_title("Reason codes in the model top-40 (observed, one slice: 2026-03, 11-day label)")
ax.set_xlabel("pages carrying the code")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig1_path = FIG_DIR / "fig_w07_reason_codes_top40.png"
fig.savefig(fig1_path, dpi=110)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.bar(arch_table["code"], arch_table["decline_pct"], color="#5878a4")
ax.axhline(base_rate, color="#a4453e", linestyle="--", label=f"base rate {base_rate:.1f}%")
ax.set_title("Decline rate per reason code vs base rate (observed, one slice)")
ax.set_ylabel("decline_11d %")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig2_path = FIG_DIR / "fig_w07_code_decline_vs_base.png"
fig.savefig(fig2_path, dpi=110)
plt.close(fig)
print("figures written:", fig1_path.name, "and", fig2_path.name, "under work/outputs/figures/")



[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


duckdb 2.0.0


sklearn 1.9.0
token found: present in environment (value not printed)


local files ready: data_0.parquet - 124.2 MB


Frame: 83,258 pages | disclosed drops: 314 absent, 37 no-pace | base rate 15.9%


Feature matrix: 83,258 rows x 21 features (identical construction to w05/w06)


receipt cross-check vs committed w05 LR row: AUC 65.9 vs 65.9 (match = True); P@20 0.55 vs 0.55 (match = True) | 36 clients, folds disjoint = True

REASON CODES - each beside the base rate it must beat (linked receipt in title):
          code     n  decline_pct  share_pct  base_rate_pct
MOMENTUM_SPIKE  5763         29.8        6.9           15.9
      CTR_LEAK 37137         19.4       44.6           15.9
 SIZE_OF_PRIZE  8328          7.2       10.0           15.9
  META_UNKNOWN 67109         15.1       80.6           15.9
receipts: MOMENTUM_SPIKE ~ w04 calibration hot-spike 29.8% n=5,763; CTR_LEAK ~ w04 gaps +11.0/+7.7/+7.3 pp;
          SIZE_OF_PRIZE ~ w04 volume table (volume OPPOSITE as risk); META_UNKNOWN ~ 67,109 pages, staleness verdict MIXED

MODEL RAIL - review-first top 40, reason codes the human sees (outcome column is printed for the honest read, NOT used to build the queue):
   reason_code  in_top40
MOMENTUM_SPIKE        37
      CTR_LEAK        35
  META_UNKNOWN        29

figures written: fig_w07_reason_codes_top40.png and fig_w07_code_decline_vs_base.png under work/outputs/figures/


## 2. Intended use and limits

**Who uses this:** the content/SEO reviewer in this lane (pseudonymous warehouse slice), deciding which pages to *open and judge first* in the coming week. The queue changes reading order; it does not act, publish, delete, or dashboard-shrink anything.

**For what:** prioritization only. At base rate 15.9%, a random 40 would hold about 6 expected decliners; the model top-40's observed count is printed in section 1 beside it (the committed receipt's headline: P@20 0.55). The reviewer treats each page as "worth a check," then applies their own context - none of which fits in the feature matrix.

**Where it stops being valid**, printed as facts, not caveats:

- One slice: month 2026-03, decision date 03-20, 11-day label window (03-21..31). Other months are unread by this notebook — the monitoring triggers in section 4 govern reuse.
- One population: 83,258 pages that passed the floor and were present in the label window (314 absent + 37 no-pace dropped, disclosed). Selection choice, not a hidden one: absent pages are unknowable this month, and the playbook says nothing about them.
- 36 clients, all folds client-disjoint — a new client needs the grouped protocol again, never a random split.
- Staleness is NOT a lever here: the committed verdict is MIXED on the 11-day horizon and 80.6% of pages have unresolvable update metadata — `META_UNKNOWN` is a lookup flag, not a risk code.
- Volume runs opposite to risk (near-floor 20.8% vs giants 7.4% decline) — which is why the size-of-prize rail exists: the model ranks risk; prize size is a human call this notebook makes explicit.
- Nothing here is causal: cross-sectional data, no intervention, no experiment. The claim ladder tops out at decision-support.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# The validity envelope, printed as facts (each anchored to the committed receipts).
print(f"slice: month=2026-03, decision date 2026-03-20, label window 03-21..31 (11 days)")
print(f"population: {len(frame):,} pages after floor + disclosed drops ({no_outcome:,} absent, {no_pace:,} no-pace)")
print(f"clients in frame: {w05['split']['clients']} | split: {w05['split']['design']} | folds client-disjoint: {disjoint_all}")
print(f"honest receipts beside every published number: AUC {queue_auc_100} / P@20 {p20_live:.2f} / P@40 {p40_obs / 100:.2f} | base rate {base_rate:.1f}%")
print("rule baseline (what to beat): AUC 0.544 / P@20 0.35 (w04 receipt)")
print(f"metadata unresolvable at decision time: {int(frame['meta_unknown'].sum()):,} pages ({100 * frame['meta_unknown'].mean():.1f}%) -> staleness verdict {w04_base['verdictA2_staleness_metadata']}")
print("volume as risk: OPPOSITE sign (w04 verdictB) - near-floor 20.8% (n=35,073) vs giants 7.4% (n=95) -> the SIZE_OF_PRIZE rail carries the biggest pages instead")
print("claim ladder status: no experiment ran -> no causal wording anywhere in this playbook; nothing here predicts a search engine")
envelope = {"slice": "2026-03, decision 2026-03-20, 11-day label", "pages": int(n_final),
            "drops": {"absent": int(no_outcome), "no_pace": int(no_pace)},
            "clients": int(w05["split"]["clients"]),
            "meta_unknown_share_pct": round(100 * float(frame["meta_unknown"].mean()), 1)}

slice: month=2026-03, decision date 2026-03-20, label window 03-21..31 (11 days)
population: 83,258 pages after floor + disclosed drops (314 absent, 37 no-pace)
clients in frame: 36 | split: client-grouped 5-fold (GroupKFold) | folds client-disjoint: True
honest receipts beside every published number: AUC 65.9 / P@20 0.55 / P@40 0.57 | base rate 15.9%
rule baseline (what to beat): AUC 0.544 / P@20 0.35 (w04 receipt)
metadata unresolvable at decision time: 67,109 pages (80.6%) -> staleness verdict MIXED
volume as risk: OPPOSITE sign (w04 verdictB) - near-floor 20.8% (n=35,073) vs giants 7.4% (n=95) -> the SIZE_OF_PRIZE rail carries the biggest pages instead
claim ladder status: no experiment ran -> no causal wording anywhere in this playbook; nothing here predicts a search engine


## 3. Human review + the no-go list

**What a person must check before acting on any row of the queue** (the queue exists so a human's attention has an order — not so a human can be replaced):

1. **Is the pace spike a real movement?** Calibration shows the hot-spike band declines more often — but seasonality, crawl-timing artifacts, or a tracking change can fake a spike. One look at the query/page timeline eitherway.
2. **Resolve `META_UNKNOWN` first.** 80.6% of the frame is unresolvable at decision time; a human with the actual CMS sees in minutes whether the page was recently maintained (according to the MIXED verdict, maintenance status was not a measured 11-day risk lever — the lookup is for context, not for triage).
3. **Confirm the `CTR_LEAK` geometry.** The code says clicks lag the band median; the human decides whether the fix is a title, a snippet, or nothing — the observed gaps are conditional on this slice's bands and label.
4. **Weigh prize size yourself.** Rails separate risk from size on purpose: a 7.4%-decline giant (n=95) can matter more than a 29.8%-decline micro-page (n=5,763 carry far less traffic each).

**The no-go list — never automate, each anchored to its receipt:**

- ❌ Auto-refresh on a staleness timer — staleness verdict MIXED (98.9% same-day visible; 80.6% post-decision unknown); the refresh play remains the *paper's* suggestion, unproven on my 11-day label.
- ❌ Auto-publish / auto-archive / auto-noindex on score alone — P@20 0.55 at base 15.9% means nearly half the top-20 did not decline; the queue names pages to open, not pages to remove.
- ❌ Client-facing "this will decline / will improve" claims — no experiment ran; causal words stay out of anything client-facing.
- ❌ Skip-page rules based on the model ("only look at ranks 1-500") — 13,037 of 13,277 decliners (98%) live beyond rank 500; blind-spot rule is part of the playbook, not an afterthought.
- ❌ Sweat the shuffle — the random-split row (AUC 67.4, P@20 wobbles 0.80–0.90 between reruns) is a labeled exhibit; any score quoted to a human comes from the grouped protocol or not at all.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Human-review checklist + the no-go list, each item anchored to its receipt.
checks = [
    {"check": "pace spike is a real movement",
     "receipt": "w04 calibration hot-spike band 29.8% vs base 15.9% (n=5,763); seasonality and crawl-timing artifacts can imitate the shape",
     "who": "reviewer reads the page's timeframe + query mix before treating MOMENTUM_SPIKE as a real draft"},
    {"check": "resolve META_UNKNOWN first",
     "receipt": f"{int(frame['meta_unknown'].sum()):,} pages ({100 * float(frame['meta_unknown'].mean()):.1f}%) unresolvable at decision time; staleness verdict {w04_base['verdictA2_staleness_metadata']} on the 11-day label",
     "who": "reviewer with real CMS access looks up the live page status in minutes - context, not triage"},
    {"check": "confirm the CTR_LEAK geometry",
     "receipt": "w04 flag_test: low-CTR decline gaps +11.0 / +7.7 / +7.3 pp in top-3 / 4-10 / 11-30, n>=30 each (observed, one slice)",
     "who": "reviewer decides title fix / snippet fix / nothing - the code marks, it does not prescribe"},
    {"check": "weigh prize size yourself",
     "receipt": "w04 volume table: near-floor 20.8% (n=35,073) vs giants 7.4% (n=95) - risk and prize are different axes",
     "who": "rail pages get human attention regardless of model score"},
]
for i, c in enumerate(checks, 1):
    print(f"{i}. human check: {c['check']}")
    print(f"   receipt: {c['receipt']}")
    print(f"   reviewer action: {c['who']}")
print()
no_go = [
    ("auto-refresh on a staleness timer",
     "staleness verdict MIXED (w04 receipts) - the refresh play stays the paper's suggestion, unproven on my 11-day label"),
    ("auto-publish / auto-archive / auto-noindex on score alone",
     f"P@20 {p20_live:.2f} at base {base_rate:.1f}% - most of the top-20 pages DID NOT decline this month"),
    ("causal wording anywhere ('will decline' / 'will improve')",
     "no experiment ran; the playbook supports decisions, not promises"),
    ("skip-page rules from model rank alone",
     f"{missed:,} of {total_dec:,} decliners ({100 * missed / max(total_dec, 1):.0f}%) rank beyond 500 - the queue is review-order, not coverage"),
    ("quoting the random-split row to humans",
     "its P@20 wobbles 0.80-0.90 between reruns; only grouped rows are claims (w06 receipt's stability note)"),
]
print("NO-GO LIST (never automate):")
for i, (item, why) in enumerate(no_go, 1):
    print(f"  x {item}")
    print(f"    because: {why}")

1. human check: pace spike is a real movement
   receipt: w04 calibration hot-spike band 29.8% vs base 15.9% (n=5,763); seasonality and crawl-timing artifacts can imitate the shape
   reviewer action: reviewer reads the page's timeframe + query mix before treating MOMENTUM_SPIKE as a real draft
2. human check: resolve META_UNKNOWN first
   receipt: 67,109 pages (80.6%) unresolvable at decision time; staleness verdict MIXED on the 11-day label
   reviewer action: reviewer with real CMS access looks up the live page status in minutes - context, not triage
3. human check: confirm the CTR_LEAK geometry
   receipt: w04 flag_test: low-CTR decline gaps +11.0 / +7.7 / +7.3 pp in top-3 / 4-10 / 11-30, n>=30 each (observed, one slice)
   reviewer action: reviewer decides title fix / snippet fix / nothing - the code marks, it does not prescribe
4. human check: weigh prize size yourself
   receipt: w04 volume table: near-floor 20.8% (n=35,073) vs giants 7.4% (n=95) - risk and prize are different a

## 4. Monitoring / retrain triggers

The playbook is valid until one of these fires; then it is rebuilt, not patched. Each trigger names the number that watches it and the number that trips it — all thresholds sit near the committed receipts, loose enough to avoid cron-noise, tight enough that drift cannot hide inside them.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# The trigger table, printed from the receipts it licenses (nothing invented):
triggers = [
    {"trigger": "new month partition lands",
     "trip_when": "always (calendar)",
     "receipt": "every metric here lived on month=2026-03, 11-day label 03-21..31",
     "action": "rebuild frame on the new month, same grouped protocol + seeds 0, revalidate before any reuse"},
    {"trigger": "grouped OOF AUC degrades",
     "trip_when": "fold mean < 0.60 (committed: 0.659, fold sd 0.019)",
     "receipt": "w05_model_receipt.json fold_auc",
     "action": "re-run the leakage checklist, inspect feature drift, retrain"},
    {"trigger": "review-first precision degrades",
     "trip_when": "P@20 < 0.40 (committed: 0.55; base rate 15.9%)",
     "receipt": "w05 comparison_table LR row",
     "action": "check fold disjointness + reason-code mix; rebuild queue"},
    {"trigger": "base-rate shift",
     "trip_when": "decline base rate outside 13-19% (committed: 15.9%)",
     "receipt": "w04/w05/w06 frame receipts agree",
     "action": "re-check label threshold (0.70x pace) against the new month before anything else"},
    {"trigger": "metadata availability changes",
     "trip_when": "meta_unknown share moves materially (committed: 80.6%)",
     "receipt": "w05 meta_unknown_pages vs frame_pages",
     "action": "if the unknown share drops, re-run the staleness tests (currently MIXED) before ever using meta_stale_days as a trigger"},
    {"trigger": "client set changes",
     "trip_when": "new client portfolio in frame (all splits stay client-grouped; 36 clients committed)",
     "receipt": "w05 split.clients / all_folds_client_disjoint",
     "action": "keep GroupKFold by client; never a random split, never client-informing features"},
]
for i, t in enumerate(triggers, 1):
    print(f"{i}. {t['trigger']} | trips when {t['trip_when']}")
    print(f"   receipt: {t['receipt']}")
    print(f"   action: {t['action']}")
print()
print("monitoring rule: thresholds sit deliberately near committed numbers - loose enough to avoid cron noise, tight enough that drift cannot hide.")

1. new month partition lands | trips when always (calendar)
   receipt: every metric here lived on month=2026-03, 11-day label 03-21..31
   action: rebuild frame on the new month, same grouped protocol + seeds 0, revalidate before any reuse
2. grouped OOF AUC degrades | trips when fold mean < 0.60 (committed: 0.659, fold sd 0.019)
   receipt: w05_model_receipt.json fold_auc
   action: re-run the leakage checklist, inspect feature drift, retrain
3. review-first precision degrades | trips when P@20 < 0.40 (committed: 0.55; base rate 15.9%)
   receipt: w05 comparison_table LR row
   action: check fold disjointness + reason-code mix; rebuild queue
4. base-rate shift | trips when decline base rate outside 13-19% (committed: 15.9%)
   receipt: w04/w05/w06 frame receipts agree
   action: re-check label threshold (0.70x pace) against the new month before anything else
5. metadata availability changes | trips when meta_unknown share moves materially (committed: 80.6%)
   receipt: w05 meta_unkno

## 5. Exports for the paper

Everything the capstone paper needs, written under `work/outputs/`:

- `action_queue_2026_03.csv` — the two-rail queue (top 40 + size-of-prize rail) with reason codes, pseudonymous ids, and the label column for the honest read. Small, locally regenerable, and git-ignored by design (`work/**/*.csv` never enters git).
- `figures/fig_w07_reason_codes_top40.png` — reason-code frequency in the top 40.
- `figures/fig_w07_code_decline_vs_base.png` — decline rate per reason code vs base rate.
- `w07_playbook_receipt.json` — the committed receipt: queue + archetype counts, triggers, no-go list, and every cross-check boolean.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Exports: queue CSV (git-ignored), two figures (committed), receipt JSON (committed).
import json

OUT_DIR = REPO / "work" / "outputs"
FIG_DIR = REPO / "work" / "outputs" / "figures"
csv_path = OUT_DIR / "action_queue_2026_03.csv"
queue_df.to_csv(csv_path, index=False)

receipt = {
    "notebook": "w07_action_playbook (ML-10, Week 7)",
    "slice": w04_sig["slice"],
    "frame_pages": int(len(frame)),
    "base_rate_pct": round(base_rate, 1),
    "disclosed_drops": {"absent_label_window": int(no_outcome), "no_pace": int(no_pace)},
    "cross_check": {"grouped_auc_vs_w05": queue_auc_100, "match_auc": bool(xc_auc),
                    "all_folds_client_disjoint": bool(disjoint_all)},
    "reason_codes": {"table": arch_rows,
                     "licenses": {
                         "MOMENTUM_SPIKE": "w04_signal_receipt test1_momentum table_main (>2.0 band 29.8%, n=5,763) + w05 top_lever/coef",
                         "CTR_LEAK": "w04_signal_receipt flag_test_ctrfix (+11.0/+7.7/+7.3 pp in top-3/4-10/11-30, n>=30 each)",
                         "SIZE_OF_PRIZE": "w04_signal_receipt test2_volume (OPPOSITE verdict; near-floor 20.8% n=35,073 vs giants 7.4% n=95)",
                         "META_UNKNOWN": "w04_baseline_receipt verdictA2_staleness_metadata MIXED + meta_unknown share 80.6%",
                         "RULE_RAIL": "size-of-prize rail: biggest pages outside model top 40"}},
    "queue": {"model_top40": {"n": int(len(top40)), "decliners": int(y[top40_idx].sum()),
                              "decline_pct_observed": round(p40_obs, 1)},
              "code_counts_top40": code_counts,
              "size_rail": {"n": int(len(rail)), "imps_sum": int(rail["imps_early"].sum())}},
    "blind_spot_rule": "13,037 of 13,277 decliners (98%) rank beyond 500 (w05 receipt); queue is review-order, not coverage",
    "no_go_list": [
        "no auto-refresh on a staleness timer (verdict MIXED; unknown-metadata share 80.6%)",
        "no auto-publish/archive/noindex on score alone (P@20 0.55 at base 15.9%)",
        "no causal wording (no experiment ran; claim ladder ends at decision-support)",
        "no skip-page rules from ranks (98% of decliners live beyond rank 500)",
        "no quoting of the random-split row to humans (wobble exhibit 0.80-0.90 P@20)",
    ],
    "monitor_triggers": triggers,
    "exports": {"queue_csv": str(csv_path.relative_to(REPO)).replace("\\", "/"),
                "figures": ["work/outputs/figures/fig_w07_reason_codes_top40.png",
                            "work/outputs/figures/fig_w07_code_decline_vs_base.png"]},
}
json_path = OUT_DIR / "w07_playbook_receipt.json"
with open(json_path, "w", encoding="utf-8") as handle:
    json.dump(receipt, handle, indent=2, default=str)
print(f"exported: {csv_path.name} ({len(queue_df)} rows) | {json_path.name} | 2 figures under work/outputs/figures/")
print("commit discipline: notebook + figures + receipt in git; the CSV stays local (work/**/*.csv is git-ignored).")

exported: action_queue_2026_03.csv (50 rows) | w07_playbook_receipt.json | 2 figures under work/outputs/figures/
commit discipline: notebook + figures + receipt in git; the CSV stays local (work/**/*.csv is git-ignored).


## Self-check

- [x] Ranked queue built from the committed receipts: model rail (top 40, honest grouped OOF) + size-of-prize rail (top 10 outside it), reason codes printed beside every row's names
- [x] Every code licensed: MOMENTUM_SPIKE (w04 calibration + w05 lever), CTR_LEAK (w04 band cells, n≥30), SIZE_OF_PRIZE (w04 volume-OPPOSITE), META_UNKNOWN (80.6% + MIXED verdict), RULE_RAIL (rail definition)
- [x] Base rate (15.9%) printed beside every rate; decliners-in-queue printed with the expected-random count
- [x] Intended use and limits stated as facts (one slice, disclosed drops, 36 clients, staleness NOT a lever, nothing causal)
- [x] Human-review checklist + no-go list, each item anchored to its receipt
- [x] Monitoring/retrain triggers table printed from committed numbers
- [x] Exports under `work/outputs/` (queue CSV local-only, two figures + receipt JSON committed)
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — public-safe wording throughout
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.